<a href="https://colab.research.google.com/github/GomezAleIvan/Coder-Data-III/blob/main/Trabajo_Final_G%C3%B3mez_Alejandro_Data_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn tensorflow

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/GomezAleIvan/Coder-Data-III/refs/heads/main/GYM.csv"
df = pd.read_csv(url)

In [ ]:
# Etiquetamos Rutina y Dieta
df['label'] = df['Exercise Schedule'].str.strip() + " | " + df['Meal Plan'].str.strip()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.compose import ColumnTransformer
import numpy as np

cat_cols = ['Gender', 'Goal', 'BMI Category']
df['text'] = df['Exercise Schedule'].fillna('') + " " + df['Meal Plan'].fillna('')

X = df[cat_cols + ['text']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# One-Hot
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Bag of Words para vectorizar el texto
bow = CountVectorizer(lowercase=True, ngram_range=(1,2), min_df=1)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

preprocess = ColumnTransformer([
    ('cat', ohe, cat_cols),
    ('bow', bow, 'text')
])

X_train_vec = preprocess.fit_transform(X_train)
X_test_vec = preprocess.transform(X_test)

classes = np.unique(y_train)
class_to_idx = {c:i for i,c in enumerate(classes)}
idx_to_class = {i:c for c,i in class_to_idx.items()}

y_train_idx = np.array([class_to_idx[c] for c in y_train])
y_test_idx = np.array([class_to_idx[c] for c in y_test])

In [6]:
# Red Neuronal
import tensorflow as tf
from tensorflow.keras import layers, models

input_dim = X_train_vec.shape[1]
num_classes = len(classes)

model = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(X_train_vec, y_train_idx,
                    validation_data=(X_test_vec, y_test_idx),
                    epochs=20,
                    batch_size=32,
                    verbose=1)

Epoch 1/20
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9939 - loss: 0.0193 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/20
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 1.0000 - loss: 1.9331e-06 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/20
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 1.0000 - loss: 4.8488e-07 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/20
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 1.0000 - loss: 2.0293e-07 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/20
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 1.0000 - loss: 5.9136e-08 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/20
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 1.0000 - loss: 1.6945e-08 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/20
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 1.0000 - loss: 1.6022e-08 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/20
2000/2000 ━━━━━━━━━━━━━━

In [7]:
# Evaluacion
test_loss, test_acc = model.evaluate(X_test_vec, y_test_idx, verbose=0)
print(f"Accuracy en test: {test_acc:.3f}")

Accuracy en test: 1.000


In [8]:
# Ejemplo caso nuevo
sample_input = pd.DataFrame([{
    'Gender': 'Male',
    'Goal': 'muscle_gain',
    'BMI Category': 'Normal weight',
    'text': 'Moderate cardio, Strength training and 5000 steps walking Balanced diet with moderate protein and carbohydrates: chicken breast brown rice spinach eggs apple'
}])

sample_vec = preprocess.transform(sample_input)
probs = model.predict(sample_vec)
pred_idx = probs.argmax(axis=1)[0]
print("Predicción:", idx_to_class[pred_idx])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
Predicción: Moderate cardio, Strength training, and 5000 steps walking | Balanced diet with moderate protein and carbohydrates: Chicken breast, brown rice, spinach, eggs, apple
